# Botrytis environmental-risk classifier

This model estimates environmental conditions favorable to gray mold; it does not detect, confirm, or predict an observed fungal infection. Classes are 0 = Low ("safety"), 1 = Moderate ("caution"), and 2 = High ("dangerous") environmental risk.

**Scientific basis:** Watanabe, H., Kojima, K., Kudomi, S., and Shimazu, T. (2021), "Estimation of greenhouse tomato infection risk by *Botrytis cinerea* based on temperature and relative humidity," *Annual Report of the Kansai Plant Protection Society*, 63, 59-65. [DOI: 10.4165/kapps.63.59](https://doi.org/10.4165/kapps.63.59). The study recorded greenhouse temperature and relative humidity every 10 minutes and evaluated a two-day cumulative effective-infection-time index with a temperature-dependent fungal-growth adjustment, reporting safety/caution/danger levels. That method is evidence that risk depends on both microclimate and exposure duration; it is not a universal threshold for every greenhouse or sensor.

Do not derive labels from guessed RH or temperature cutoffs. Supply a timestamped `environmental_risk_labeled.csv` whose `environmental_risk_label` and `rule_based_risk_label` were created under a documented, approved protocol. Set the rule descriptions in the first code cell, using exact published parameters or an explicitly reviewed local environmental-risk rule. These labels represent environmental risk only, not confirmed disease. The notebook does not download data or synthesize labels.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

REFERENCE = "Watanabe et al. (2021), DOI:10.4165/kapps.63.59"
RISK_LABEL_RULE_DESCRIPTION = None
RULE_BASELINE_DESCRIPTION = None
DURATION_FEATURE_DESCRIPTION = None

CLASS_IDS = np.array([0, 1, 2], dtype=np.int64)
CLASS_NAMES = ["Low environmental risk", "Moderate environmental risk", "High environmental risk"]
BASE_FEATURE_COLUMNS = [
    "temperature_c",
    "relative_humidity_pct",
    "vpd_kpa",
    "dew_point_margin_c",
]
OPTIONAL_DURATION_COLUMN = "sustained_high_humidity_minutes"

if not RISK_LABEL_RULE_DESCRIPTION or not RULE_BASELINE_DESCRIPTION:
    raise ValueError(
        "Set RISK_LABEL_RULE_DESCRIPTION and RULE_BASELINE_DESCRIPTION only "
        "after documenting the source-backed environmental-risk labeling protocol."
    )

repo_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
REPO_ROOT = next(
    (path for path in repo_candidates
     if (path / "model_training" / "preprocess_greenhouse.py").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the OmniNode_AI repository.")

DATA_PATH = REPO_ROOT / "data" / "environmental_risk_labeled.csv"
ARTIFACT_DIR = REPO_ROOT / "model_training" / "artifacts"
if not DATA_PATH.is_file():
    raise FileNotFoundError(
        "No labeled environmental-risk data found at {}. Join approved labels "
        "to processed sensor records; this notebook will not invent labels.".format(DATA_PATH)
    )

frame = pd.read_csv(DATA_PATH)
feature_columns = list(BASE_FEATURE_COLUMNS)
if OPTIONAL_DURATION_COLUMN in frame.columns:
    if not DURATION_FEATURE_DESCRIPTION:
        raise ValueError(
            "Document the source and computation for sustained high-humidity "
            "duration before using it as a feature."
        )
    feature_columns.append(OPTIONAL_DURATION_COLUMN)
required_columns = [
    "timestamp_utc",
    *feature_columns,
    "environmental_risk_label",
    "rule_based_risk_label",
]
missing_columns = [name for name in required_columns if name not in frame.columns]
if missing_columns:
    raise ValueError("Labeled CSV is missing columns: {}".format(missing_columns))

timestamp_text = frame["timestamp_utc"].astype(str)
if not timestamp_text.str.endswith(("Z", "+00:00")).all():
    raise ValueError("timestamp_utc values must be explicitly normalized to UTC.")
frame["timestamp_utc"] = pd.to_datetime(timestamp_text, utc=True, errors="raise")
for column in feature_columns + ["environmental_risk_label", "rule_based_risk_label"]:
    frame[column] = pd.to_numeric(frame[column], errors="coerce")
frame = frame.sort_values("timestamp_utc").reset_index(drop=True)
if frame["timestamp_utc"].duplicated().any():
    raise ValueError("Labeled CSV contains duplicate timestamps.")

valid_rows = frame[feature_columns + ["environmental_risk_label", "rule_based_risk_label"]].notna().all(axis=1)
dropped_rows = int((~valid_rows).sum())
frame = frame.loc[valid_rows].reset_index(drop=True)
for column in ("environmental_risk_label", "rule_based_risk_label"):
    if not frame[column].isin(CLASS_IDS).all():
        raise ValueError("{} must contain only class IDs 0, 1, and 2.".format(column))
    frame[column] = frame[column].astype(np.int64)
if not np.isfinite(frame[feature_columns].to_numpy(dtype=np.float32)).all():
    raise ValueError("Feature values must be finite; do not impute missing measurements.")
if OPTIONAL_DURATION_COLUMN in feature_columns and (frame[OPTIONAL_DURATION_COLUMN] < 0).any():
    raise ValueError("Sustained high-humidity duration cannot be negative.")
print("Loaded {} labeled readings; dropped {} rows with missing features/labels.".format(
    len(frame), dropped_rows
))
print("Classes are environmental risk estimates, not disease observations.")


In [ ]:
row_count = len(frame)
train_stop = int(row_count * 0.70)
validation_stop = int(row_count * 0.85)
if not (0 < train_stop < validation_stop < row_count):
    raise ValueError("Not enough chronological rows for 70/15/15 data splits.")

train_frame = frame.iloc[:train_stop]
validation_frame = frame.iloc[train_stop:validation_stop]
test_frame = frame.iloc[validation_stop:]
X_train_raw = train_frame[feature_columns].to_numpy(dtype=np.float32)
X_validation_raw = validation_frame[feature_columns].to_numpy(dtype=np.float32)
X_test_raw = test_frame[feature_columns].to_numpy(dtype=np.float32)
y_train = train_frame["environmental_risk_label"].to_numpy(dtype=np.int64)
y_validation = validation_frame["environmental_risk_label"].to_numpy(dtype=np.int64)
y_test = test_frame["environmental_risk_label"].to_numpy(dtype=np.int64)
y_rule_test = test_frame["rule_based_risk_label"].to_numpy(dtype=np.int64)
if not np.array_equal(np.unique(y_train), CLASS_IDS):
    raise ValueError("The chronological training period must contain all three risk classes.")

x_mean = X_train_raw.mean(axis=0)
x_scale = X_train_raw.std(axis=0)
x_scale = np.where(x_scale == 0.0, 1.0, x_scale)
X_train_scaled = ((X_train_raw - x_mean) / x_scale).astype(np.float32)
X_validation_scaled = ((X_validation_raw - x_mean) / x_scale).astype(np.float32)
X_test_scaled = ((X_test_raw - x_mean) / x_scale).astype(np.float32)

print("Chronological records - train: {}, validation: {}, test: {}".format(
    len(X_train_raw), len(X_validation_raw), len(X_test_raw)
))
print("Feature scaler fitted on training records only:", feature_columns)


In [ ]:
tf.keras.utils.set_random_seed(42)
classifier = keras.Sequential(
    [
        keras.Input(shape=(len(feature_columns),)),
        layers.Dense(16, activation="relu"),
        layers.Dense(8, activation="relu"),
        layers.Dense(len(CLASS_IDS), activation="softmax"),
    ]
)
classifier.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = classifier.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_validation_scaled, y_validation),
    epochs=100,
    batch_size=32,
    shuffle=False,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True
        )
    ],
    verbose=1,
)


In [ ]:
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support


def evaluate_predictions(title, y_true, y_pred):
    matrix = confusion_matrix(y_true, y_pred, labels=CLASS_IDS)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASS_IDS, zero_division=0
    )
    print("\n{}".format(title))
    print("Confusion matrix (rows=true, columns=predicted; IDs 0,1,2):")
    print(matrix)
    for index, name in enumerate(CLASS_NAMES):
        print("{}: precision={:.4f}, recall={:.4f}, F1={:.4f}, support={}".format(
            name, precision[index], recall[index], f1[index], support[index]
        ))
    macro = precision_recall_fscore_support(
        y_true, y_pred, labels=CLASS_IDS, average="macro", zero_division=0
    )
    print("Macro precision={:.4f}, recall={:.4f}, F1={:.4f}".format(
        macro[0], macro[1], macro[2]
    ))
    return {"confusion_matrix": matrix, "precision": precision, "recall": recall, "f1": f1}


float_probabilities = classifier.predict(X_test_scaled, verbose=0)
y_test_prediction_float = np.argmax(float_probabilities, axis=1)
float_metrics = evaluate_predictions(
    "Float model: chronological test-set metrics", y_test, y_test_prediction_float
)
rule_metrics = evaluate_predictions(
    "Documented rule baseline: same chronological test records", y_test, y_rule_test
)


In [ ]:
import json

representative_indices = np.linspace(
    0, len(X_train_scaled) - 1,
    num=min(128, len(X_train_scaled)),
    dtype=int,
)

def representative_dataset():
    for index in representative_indices:
        yield [X_train_scaled[index : index + 1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(classifier)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
int8_model = converter.convert()

interpreter = tf.lite.Interpreter(model_content=int8_model)
interpreter.allocate_tensors()
input_detail = interpreter.get_input_details()[0]
output_detail = interpreter.get_output_details()[0]
if input_detail["dtype"] != np.int8 or output_detail["dtype"] != np.int8:
    raise RuntimeError("Converter did not produce fully INT8 input and output tensors.")
if tuple(input_detail["shape"]) != (1, len(feature_columns)):
    raise RuntimeError("Unexpected INT8 classifier input shape: {}".format(input_detail["shape"]))
if tuple(output_detail["shape"]) != (1, len(CLASS_IDS)):
    raise RuntimeError("Unexpected INT8 classifier output shape: {}".format(output_detail["shape"]))

input_scale, input_zero_point = input_detail["quantization"]
output_scale, output_zero_point = output_detail["quantization"]
if input_scale <= 0.0 or output_scale <= 0.0:
    raise RuntimeError("INT8 input/output quantization scales must be positive.")

def quantize_tensor(values, detail):
    scale, zero_point = detail["quantization"]
    limits = np.iinfo(detail["dtype"])
    quantized = np.rint(values / scale + zero_point)
    return np.clip(quantized, limits.min, limits.max).astype(detail["dtype"])

int8_probabilities = []
for sample in X_test_scaled:
    interpreter.set_tensor(
        input_detail["index"], quantize_tensor(sample[np.newaxis, :], input_detail)
    )
    interpreter.invoke()
    output = interpreter.get_tensor(output_detail["index"]).astype(np.float32)
    int8_probabilities.append((output - output_zero_point) * output_scale)
y_test_prediction_int8 = np.argmax(np.asarray(int8_probabilities), axis=1)
int8_metrics = evaluate_predictions(
    "Full-INT8 model: chronological test-set metrics", y_test, y_test_prediction_int8
)
evaluate_predictions(
    "Documented rule baseline: same chronological test records", y_test, y_rule_test
)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "omninode_risk_classifier_int8.tflite"
header_path = ARTIFACT_DIR / "omninode_risk_classifier_int8.h"
metadata_path = ARTIFACT_DIR / "risk_classifier_preprocessing.json"
model_bytes = bytes(int8_model)
model_path.write_bytes(model_bytes)
byte_lines = [
    "  " + ", ".join("0x{:02x}".format(value) for value in model_bytes[start : start + 12])
    for start in range(0, len(model_bytes), 12)
]
header = "\n".join(
    [
        "#pragma once",
        "#include <cstddef>",
        "",
        "alignas(16) const unsigned char omninode_risk_classifier_int8[] = {",
        ",\n".join(byte_lines),
        "};",
        "constexpr std::size_t omninode_risk_classifier_int8_len = sizeof(omninode_risk_classifier_int8);",
        "",
    ]
)
header_path.write_text(header, encoding="utf-8")
metadata = {
    "feature_order": feature_columns,
    "feature_mean": [float(value) for value in x_mean],
    "feature_scale": [float(value) for value in x_scale],
    "class_mapping": {"0": "low environmental risk", "1": "moderate environmental risk", "2": "high environmental risk"},
    "label_rule_description": RISK_LABEL_RULE_DESCRIPTION,
    "rule_baseline_description": RULE_BASELINE_DESCRIPTION,
    "duration_feature_description": DURATION_FEATURE_DESCRIPTION,
    "reference": REFERENCE,
    "input_quantization": {"scale": float(input_scale), "zero_point": int(input_zero_point)},
    "output_quantization": {"scale": float(output_scale), "zero_point": int(output_zero_point)},
    "source_data": str(DATA_PATH.relative_to(REPO_ROOT)),
}
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print("Full-INT8 model bytes: {}".format(len(model_bytes)))
print("Candidate artifacts: {}, {}, {}".format(model_path, header_path, metadata_path))
